# P4c - the real run (Qwen3.5-4B, bf16 LoRA, 7,800 samples, 1 epoch)

Follows `p4a_finetune_smoke.ipynb` (plumbing, 120 samples, 50 steps) and
`p4a_verify_e0_e1.ipynb` (E0 rendering fix, E1 resolution pricing). Context:
`ai-collab/handover-vlm-parser.md` section 6.

**Budget, measured rather than guessed.** P4a measured 7.54 s/step at an effective
batch of 8 on an L4, and the L4 bills 1.54 compute units/hour. 7,800 training samples
is 975 steps, so roughly **2.04 hours and 3.2 CU**. The held-out inference pass on top
of that has **not** been measured - section 11 times five samples and extrapolates before
committing to all 200.

**What this notebook does *not* do:** compute the metrics. It writes raw model output
to `p4c_holdout_predictions.jsonl` on Drive, and scoring happens back in the repo with
`uv run python -m src.core.vl_models.score_predictions`. That is deliberate - the
project already paid once for benchmark code and shipped parser code drifting apart,
and a metric reimplemented in a notebook is exactly how it happens again.

**Scope, and what the result will and will not prove.** Training and evaluation are
both on our own renderer. A good number here means "it learned to read the images we
draw"; it does **not** mean "it can read a LinkedIn screenshot". That would need P3
(30-50 real screenshots, hand-labelled), which is deliberately deferred.

The runtime bills while connected, idle included. `Runtime -> Disconnect and delete
runtime` when finished.


## 1. Install (several minutes)

Verbatim from `p4a_finetune_smoke.ipynb`, which took it verbatim from the official
Unsloth `Qwen3_5_(4B)_Vision.ipynb`. Do **not** "modernise" the pins: the stack is
built against torch 2.8.0 while Colab defaults to something newer, and the CUDA
extensions only build against the pinned one.

`%%capture` is deliberately absent. Upstream uses it to keep the notebook tidy; here a
silently failed install would resurface an hour later as a confusing `import unsloth`
error.


In [1]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 142.9 MB/s eta 0:00:0000:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 131ms                                         
Prepared 1 package in 77ms                                               
Uninstalled 1 package in 3ms
Installed 1 package in 10ms                                 
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Resolved 28 packages in 124ms                                        
Prepared 2 packages in 463ms                                             
Uninstalled 1 package in 75ms
Installed 2 packages in 49ms                                
 - transformers==5.5.0
 + transformers==5.2.0
 + typer-slim==0.24.0
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 184ms                                        
Prepared 4 packages in 13.54s                                            
Installed 4 packages in 16ms                                
 + causal-co

## 2. Refuse the wrong runtime


In [2]:
# Refuse the wrong runtime here rather than discover it an hour into a run.
import torch

capability = torch.cuda.get_device_capability(0)
print("device       ", torch.cuda.get_device_name(0))
print("capability   ", capability)
print("VRAM GiB     ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("bf16 NATIVE  ", torch.cuda.is_bf16_supported(including_emulation=False))

# `is_bf16_supported()` defaults to including_emulation=True and answers True on a
# Turing T4, which has no bf16 hardware at all. Always ask for the native answer.
assert capability >= (8, 0), (
    f"Got capability {capability}; this needs Ampere or newer (L4 is 8.9). "
    "Runtime -> Disconnect and delete runtime, then reconnect having picked L4."
)


device        NVIDIA L4
capability    (8, 9)
VRAM GiB      22.03
bf16 NATIVE   True


## 3. Verify the upload, then copy it off Drive

If `drive.mount` fails with `credentials-propagation ... Bad Request`, open a Colab tab
in the browser for this same session and re-run this cell. The authorisation flow needs
somewhere to appear, and driving the kernel from VS Code gives it nowhere.


In [4]:
import hashlib
import json
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/colab_finetune")
ARCHIVE = "zip_vl_6x6_8000_20260822.tar"
DATASET = "main_6x6"
EXPECTED_SHA256 = "69c753e16d030ecba7a7da505f046b747e275ec7ca956c0bada80a6d7ce70fbf"

drive.mount("/content/drive")

source = DRIVE_DIR / ARCHIVE
digest = hashlib.sha256(source.read_bytes()).hexdigest()
print("sha256   ", digest)
print("expected ", EXPECTED_SHA256)
# The dataset, not the build command, is the unit of reproducibility: generate_puzzle
# aborts its search on wall-clock time, so rebuilding from the same seed does NOT give
# the same bytes. This digest is what says the copy up here is the copy checked locally.
assert digest == EXPECTED_SHA256, "Upload is corrupt, or this is the wrong archive."

# Copy onto the VM's own disk before reading. Pulling thousands of small files through
# the Drive FUSE mount throttles the dataloader badly.
!cp "{source}" /content/
!tar -xf /content/{ARCHIVE} -C /content/
!ls /content/{DATASET} && ls /content/{DATASET}/images | wc -l


Mounted at /content/drive
sha256    69c753e16d030ecba7a7da505f046b747e275ec7ca956c0bada80a6d7ce70fbf
expected  69c753e16d030ecba7a7da505f046b747e275ec7ca956c0bada80a6d7ce70fbf
images	manifest.json  metadata.jsonl
8000


## 4. Split off the held-out set

**The 120-sample `smoke_6x6` archive P4a used is NOT a valid held-out set for this
run.** `dataset_builder.draw_recipe` seeds every item with `random.Random(seed + index)`,
and the two packs were built one apart - 20260822 and 20260823 - so `smoke[i]` is
`main[i+1]`. Checked locally on 2026-08-22: **all 120** share an identical render recipe
(theme, cell size, rotation, JPEG quality) with a sample in this archive and **82 of 120**
have an identical label. Only the wall-clock non-determinism in the generator kept the
other 38 apart. Evaluating on it would be scoring the model on its own training data.

So the held-out set is carved out here instead, from the tail of the same archive, and
excluded from training. Same file, same digest, nothing extra to upload.


In [5]:
from collections import Counter

DATA_DIR = Path(f"/content/{DATASET}")
HOLDOUT_SIZE = 200

# Verbatim from src/core/vl_models/prompt_variants.FINETUNE_INSTRUCTION.
# Train and infer with the SAME string -- a checkpoint queried with the baseline
# few-shot prompt is being asked a question it never saw.
INSTRUCTION = 'Read this Zip puzzle screenshot and reply with ONLY a JSON object.\n"layout" is a 2D array of two-character strings: "  " for an empty cell, "xx" for a blocked cell, and a zero-padded number such as "01" for a waypoint.\n"walls" is a list of {"cell1": [row, col], "cell2": [row, col]} objects, one per thick black bar drawn on a grid line between two neighbouring cells. Report every wall you can see and do not invent any.'

records = [
    json.loads(line)
    for line in (DATA_DIR / "metadata.jsonl").read_text("utf-8").splitlines()
]
train_records, holdout = records[:-HOLDOUT_SIZE], records[-HOLDOUT_SIZE:]

assert not ({record["file_name"] for record in train_records}
            & {record["file_name"] for record in holdout})

print(f"{len(records)} records -> train {len(train_records)}, holdout {len(holdout)}")
print("holdout walls per board:", dict(sorted(Counter(r["wall_count"] for r in holdout).items())))
print("holdout themes         :", dict(Counter(r["theme"] for r in holdout)))


8000 records -> train 7800, holdout 200
holdout walls per board: {0: 11, 1: 16, 2: 18, 3: 15, 4: 12, 5: 15, 6: 13, 7: 16, 8: 15, 9: 20, 10: 16, 11: 9, 12: 24}
holdout themes         : {'dark': 44, 'light': 156}


## 5. A lazy dataset

**Do not reuse P4a's loader here.** It decoded every image into a PIL object in a Python
list; at roughly 1.2 MB decoded that is about 10 GB for 8,000 images against the ~12.7 GB
a standard Colab VM has. Fine at 120 samples, fatal at 8,000.

`datasets.Dataset.from_list` over the *metadata only* (two short strings per row) plus
`set_transform` opens each image at access time, so memory is bounded by the batch rather
than by the dataset.

`imagefolder` would be the obvious alternative and is the wrong tool here twice over: it
reads `images/` as a class directory, and its automatic `label` column collides with the
`label` column this dataset already has.


In [6]:
from datasets import Dataset
from PIL import Image


def to_messages(file_name: str, label: str) -> list[dict]:
    """One training conversation. The image is opened here, not at build time."""
    return [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": INSTRUCTION},
                {"type": "image", "image": Image.open(DATA_DIR / file_name).convert("RGB")},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": label}]},
    ]


def transform(batch: dict) -> dict:
    return {
        "messages": [
            to_messages(file_name, label)
            for file_name, label in zip(batch["file_name"], batch["label"])
        ]
    }


train_dataset = Dataset.from_list(
    [{"file_name": r["file_name"], "label": r["label"]} for r in train_records]
)
train_dataset.set_transform(transform)

print(len(train_dataset), "training rows")
print(train_dataset[0]["messages"][0]["content"][1]["image"])


7800 training rows
<PIL.Image.Image image mode=RGB size=564x632 at 0x786C95A54800>


### Does it actually stay lazy?

The claim above is worth one measurement rather than a comment. Resident memory should
barely move across a few hundred accesses; if it climbs by hundreds of MB, something is
holding references and the long run will die.


In [7]:
import gc

import psutil

PROBE_SAMPLES = 300
DECODED_IMAGE_MB = 1.2

process = psutil.Process()
gc.collect()
before = process.memory_info().rss / 1024**2
for index in range(PROBE_SAMPLES):
    row = train_dataset[index]
    assert row["messages"][0]["content"][1]["image"].mode == "RGB"
gc.collect()
after = process.memory_info().rss / 1024**2

print(f"RSS before {before:8.1f} MB")
print(f"RSS after  {after:8.1f} MB   ({after - before:+.1f} MB over {PROBE_SAMPLES} samples)")
print(f"total RAM  {psutil.virtual_memory().total / 1024**3:.1f} GiB")
print()
print(f"Materialised, this probe alone would hold ~{PROBE_SAMPLES * DECODED_IMAGE_MB:.0f} MB")
print(f"and the full set ~{len(train_dataset) * DECODED_IMAGE_MB / 1024:.1f} GB.")


RSS before    744.2 MB
RSS after     753.9 MB   (+9.7 MB over 300 samples)
total RAM  53.0 GiB

Materialised, this probe alone would hold ~360 MB
and the full set ~9.1 GB.


## 6. Model and LoRA


In [8]:
from unsloth import FastVisionModel

# load_in_4bit=False -> 16-bit LoRA. Unsloth advises against QLoRA for Qwen3.5 (larger
# than usual quantisation error), which is why this route needs native bf16 and therefore
# a paid L4 rather than the free T4.
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)


/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.12/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [9]:
model = FastVisionModel.get_peft_model(
    model,
    # The failure this project is fixing is purely visual: numbers and layout are already
    # read correctly untuned (cell accuracy 0.961) and what is missed are the black bars
    # pressed onto the grid lines (wall F1 0.438). Freezing the vision tower to save
    # memory would very likely learn nothing that matters here. P4a confirmed the vision
    # layers do move, and move more than the language layers.
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 7. Smoke the pipeline before spending two hours on it

Five real steps through the real collator, at **learning rate 0**. Everything is
exercised - dataloader workers, lazy decode, collation, forward, backward, optimizer -
while the weights provably do not move, so this costs a minute and contaminates nothing.
The assertion at the end is what makes "provably" more than a claim.

What to look at: the measured seconds/step against P4a's 7.54, since lazy decoding is new
here, and peak VRAM against the 22.03 GiB an L4 has. If the dataloader workers misbehave,
this is where it shows - set `DATALOADER_WORKERS = 0` and re-run the cell.

**Not verified:** this notebook has never been executed, so building a second `SFTTrainer`
over the same model in section 8 is expected to work but has not been shown to. If it
raises, the dry run has already told you what it was for - restart the runtime, re-run
sections 1-6, and go straight to section 8.


In [10]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
EFFECTIVE_BATCH = PER_DEVICE_BATCH * GRAD_ACCUM
DATALOADER_WORKERS = 2
MAX_LENGTH = 2048          # Longest label in the set is 769 characters (~250 tokens),
                           # so image tokens dominate this budget, not the answer.
DRY_RUN_STEPS = 5
L4_COMPUTE_UNITS_PER_HOUR = 1.54

FastVisionModel.for_training(model)

sample_lora_b = next(
    parameter for name, parameter in model.named_parameters()
    if parameter.requires_grad and "lora_B" in name and ".visual." in name
)
before_dry_run = sample_lora_b.detach().clone()

dry_run = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        max_steps = DRY_RUN_STEPS,
        learning_rate = 0.0,       # the whole point: exercise everything, change nothing
        warmup_steps = 0,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "constant",
        seed = 3407,
        output_dir = "/content/dry_run",
        save_strategy = "no",
        report_to = "none",
        dataloader_num_workers = DATALOADER_WORKERS,
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = MAX_LENGTH,
    ),
)
dry_stats = dry_run.train()

assert torch.equal(before_dry_run, sample_lora_b), (
    "The dry run moved the weights; it was supposed to run at learning rate 0."
)

dry_seconds_per_step = dry_stats.metrics["train_runtime"] / DRY_RUN_STEPS
projected_hours = len(train_dataset) / EFFECTIVE_BATCH * dry_seconds_per_step / 3600
print()
print("weights unchanged: True")
print(f"measured   {dry_seconds_per_step:.2f} s/step  (P4a, images pre-decoded in RAM: 7.54)")
print(f"peak VRAM  {torch.cuda.max_memory_reserved() / 1024**3:.2f}"
      f" / {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")
print(f"projected  {projected_hours:.2f} h for 1 epoch"
      f" (~{projected_hours * L4_COMPUTE_UNITS_PER_HOUR:.1f} compute units at the L4 rate)")


Unsloth: Model does not have a default image size - using 512
Unsloth: Your learning rate of `0.0` is too small and less than 1e-7! Consider increasing it, otherwise gradient updates will be close to 0!


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,800 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 38,756,352 of 4,578,021,888 (0.85% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.091991
2,0.928068
3,1.133920
4,0.955536
5,0.956622



weights unchanged: True
measured   37.59 s/step  (P4a, images pre-decoded in RAM: 7.54)
peak VRAM  13.93 / 22.03 GiB
projected  10.18 h for 1 epoch (~15.7 compute units at the L4 rate)


## 8. Train

One epoch, 975 steps at an effective batch of 8. Two differences from P4a beyond the
sample count:

- **`warmup_steps = 30`** rather than 5. Five was 10% of a 50-step test; 30 is the usual
  ~3% of a 975-step run.
- **Checkpoints go to Drive.** Two hours is long enough to lose a session to a
  disconnect. `save_steps = 200` is a deliberate trade: writing ~350 MB through the Drive
  FUSE mount is slow, so saving more often would spend a real slice of the run on I/O,
  while saving less often risks losing more than half an hour of work. `save_total_limit`
  keeps the last two so Drive does not fill up.

**If the session drops**, re-run the notebook from the top and set `RESUME = True` in the
next cell. `resume_from_checkpoint` needs the same seed and the same dataset order to be
meaningful, and both are fixed here.


In [11]:
RESUME = False   # set True after a disconnect to continue from the newest checkpoint

CHECKPOINT_DIR = DRIVE_DIR / "p4c_qwen35_4b_zip_checkpoints"
SAVE_STEPS = 200
WARMUP_STEPS = 30
LEARNING_RATE = 2e-4

existing = sorted(CHECKPOINT_DIR.glob("checkpoint-*")) if CHECKPOINT_DIR.is_dir() else []
print("checkpoints already on Drive:", [path.name for path in existing] or "none")
assert not (RESUME and not existing), "RESUME is set but there is nothing to resume from."

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs = 1,
        warmup_steps = WARMUP_STEPS,
        learning_rate = LEARNING_RATE,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = str(CHECKPOINT_DIR),
        save_steps = SAVE_STEPS,
        save_total_limit = 2,
        report_to = "none",
        dataloader_num_workers = DATALOADER_WORKERS,
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = MAX_LENGTH,
    ),
)
print(f"{len(train_dataset) // EFFECTIVE_BATCH} optimizer steps expected")


checkpoints already on Drive: none
Unsloth: Model does not have a default image size - using 512
975 optimizer steps expected


In [12]:
trainer_stats = trainer.train(resume_from_checkpoint=RESUME or None)


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,800 | Num Epochs = 1 | Total steps = 975
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 38,756,352 of 4,578,021,888 (0.85% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,0.901588
20,0.378245
30,0.030484
40,0.009810
50,0.005689
60,0.002926
70,0.001774
80,0.001701
90,0.000565
100,0.000539


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_checkpoints/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_checkpoints/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_checkpoints/checkpoint-975/tokenizer_config.json.


## 9. What it cost, and did the vision tower move


In [13]:
runtime = trainer_stats.metrics["train_runtime"]
steps = trainer_stats.global_step
peak = torch.cuda.max_memory_reserved() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3

print(f"steps      {steps}")
print(f"runtime    {runtime / 3600:.2f} h  ({runtime / steps:.2f} s/step)")
print(f"cost       ~{runtime / 3600 * L4_COMPUTE_UNITS_PER_HOUR:.1f} compute units")
print(f"peak VRAM  {peak:.2f} / {total:.2f} GiB  ({peak / total * 100:.1f}%)")
print(f"final loss {trainer_stats.metrics.get('train_loss')}")


steps      975
runtime    1.56 h  (5.77 s/step)
cost       ~2.4 compute units
peak VRAM  20.90 / 22.03 GiB  (94.9%)
final loss 0.013818830417766525


In [14]:
# The bottleneck is visual, so "did the vision layers actually learn anything" is not a
# rhetorical question. P4a saw max|B| 0.114 in the visual stack against 0.059 in the
# language stack after 50 steps; a full epoch should be clearly larger.
visual, language = [], []
for name, parameter in model.named_parameters():
    if parameter.requires_grad and "lora_B" in name:
        (visual if ".visual." in name else language).append(parameter)

for label, group in (("visual", visual), ("language", language)):
    if not group:
        print(f"{label:9s}: NO lora_B")
        continue
    nonzero = sum(1 for parameter in group if parameter.abs().max().item() > 0)
    largest = max(parameter.abs().max().item() for parameter in group)
    print(f"{label:9s}: {nonzero}/{len(group)} lora_B non-zero, max|B|={largest:.3e}")


visual   : 96/96 lora_B non-zero, max|B|=2.723e-01
language : 248/248 lora_B non-zero, max|B|=1.662e-01


## 10. Save the adapter


In [15]:
# The LoRA adapter only. Merged 16-bit and GGUF exports are P4d, and GGUF has a known
# vision-export defect (unsloth#3899) that needs checking before it is relied on.
ADAPTER_OUT = DRIVE_DIR / "p4c_qwen35_4b_zip_lora"
model.save_pretrained(str(ADAPTER_OUT))
tokenizer.save_pretrained(str(ADAPTER_OUT))
!du -sh "{ADAPTER_OUT}"


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_lora/tokenizer_config.json.


168M	/content/drive/MyDrive/colab_finetune/p4c_qwen35_4b_zip_lora


## 11. Held-out inference

`build_inference_prompt` renders a conversation through the **same** template call
training uses and cuts at the answer, so the prompt prefix matches training byte for
byte instead of relying on inference-side arguments being kept in sync by hand. P4a
proved how much this matters: the same adapter scored JSON 4/4 with it and 0/3 without,
on identical images - the content was right, the shape was not.

Note `enable_thinking=False` would **not** be equivalent: it emits `<think>\n\n</think>\n\n`,
which training never saw either.

Nothing is scored here. Each answer is written raw to Drive, line by line, so a
disconnect keeps whatever finished.


In [16]:
import time


def build_inference_prompt(tokenizer, instruction: str) -> str:
    """Render through the SAME template call training uses, then cut at the answer."""
    sentinel = "@@ANSWER@@"
    conversation = [
        {"role": "user", "content": [{"type": "text", "text": instruction},
                                     {"type": "image"}]},
        {"role": "assistant", "content": [{"type": "text", "text": sentinel}]},
    ]
    return tokenizer.apply_chat_template(conversation, tokenize=False).split(sentinel)[0]


MAX_NEW_TOKENS = 700       # longest label is 769 characters, ~250 tokens
PROGRESS_EVERY = 25


def run_predictions(records: list[dict], out_path: Path) -> float:
    """Greedy-decodes each record into a JSONL. Returns seconds per sample."""
    FastVisionModel.for_inference(model)
    prompt = build_inference_prompt(tokenizer, INSTRUCTION)
    started = time.perf_counter()

    with out_path.open("w", encoding="utf-8") as handle:
        for position, record in enumerate(records, start=1):
            image = Image.open(DATA_DIR / record["file_name"]).convert("RGB")
            inputs = tokenizer(image, prompt, add_special_tokens=False,
                               return_tensors="pt").to("cuda")
            call_started = time.perf_counter()
            generated = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                       use_cache=True, do_sample=False)
            elapsed = time.perf_counter() - call_started
            raw = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:],
                                   skip_special_tokens=True)
            handle.write(json.dumps({**record, "raw_output": raw,
                                     "generation_seconds": round(elapsed, 3)}) + "\n")
            handle.flush()
            if position % PROGRESS_EVERY == 0:
                print(f"  {position}/{len(records)}")

    return (time.perf_counter() - started) / len(records)


print("prompt ends with:", repr(build_inference_prompt(tokenizer, INSTRUCTION)[-64:]))


prompt ends with: 'vision_end|><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


### First five, to price the pass

If this looks wrong - no JSON, or a minute per sample - stop here rather than paying for
another 195 of the same.


In [17]:
PROBE = 5
PROBE_OUT = Path("/content/probe_predictions.jsonl")

seconds_per_sample = run_predictions(holdout[:PROBE], PROBE_OUT)
probe = [json.loads(line) for line in PROBE_OUT.read_text("utf-8").splitlines()]
json_shaped = sum(1 for r in probe if r["raw_output"].lstrip().startswith(("{", "```")))

print(f"\n{seconds_per_sample:.1f} s/sample -> "
      f"{seconds_per_sample * len(holdout) / 60:.0f} min for all {len(holdout)}")
print(f"looks like JSON: {json_shaped}/{PROBE}")
print("\n--- ground truth ---")
print(probe[0]["label"])
print("\n--- model ---")
print(probe[0]["raw_output"][:800])



38.5 s/sample -> 128 min for all 200
looks like JSON: 5/5

--- ground truth ---
{
  "layout": [
    ["  ", "06", "  ", "  ", "  ", "  "],
    ["  ", "  ", "  ", "09", "08", "  "],
    ["  ", "  ", "07", "  ", "  ", "01"],
    ["  ", "  ", "  ", "  ", "  ", "  "],
    ["05", "  ", "  ", "02", "  ", "03"],
    ["  ", "04", "  ", "  ", "  ", "  "]
  ],
  "walls": [
    {"cell1": [0, 4], "cell2": [1, 4]},
    {"cell1": [1, 0], "cell2": [1, 1]},
    {"cell1": [1, 1], "cell2": [1, 2]},
    {"cell1": [1, 2], "cell2": [2, 2]},
    {"cell1": [2, 1], "cell2": [3, 1]},
    {"cell1": [2, 3], "cell2": [3, 3]},
    {"cell1": [2, 4], "cell2": [2, 5]},
    {"cell1": [2, 4], "cell2": [3, 4]},
    {"cell1": [3, 5], "cell2": [4, 5]},
    {"cell1": [4, 3], "cell2": [5, 3]},
    {"cell1": [4, 4], "cell2": [5, 4]}
  ]
}

--- model ---
{
  "layout": [
    ["  ", "06", "  ", "  ", "  ", "  "],
    ["  ", "  ", "  ", "09", "08", "  "],
    ["  ", "  ", "07", "  ", "  ", "01"],
    ["  ", "  ", "  ", "  ", "  

### All 200


In [18]:
PREDICTIONS_OUT = DRIVE_DIR / "p4c_holdout_predictions.jsonl"

seconds_per_sample = run_predictions(holdout, PREDICTIONS_OUT)
written = len(PREDICTIONS_OUT.read_text("utf-8").splitlines())

print(f"\n{written}/{len(holdout)} written to {PREDICTIONS_OUT}")
print(f"{seconds_per_sample:.1f} s/sample, {seconds_per_sample * written / 60:.0f} min total")


  25/200
  50/200
  75/200
  100/200
  125/200
  150/200
  175/200
  200/200

200/200 written to /content/drive/MyDrive/colab_finetune/p4c_holdout_predictions.jsonl
34.5 s/sample, 115 min total


## 12. A parse-rate glance, and then get out

The only thing worth checking before disconnecting is whether the answers are the right
*shape*, because that is the one failure a re-run could still fix cheaply. Accuracy is
scored locally.


In [19]:
import re

predictions = [json.loads(line) for line in
               PREDICTIONS_OUT.read_text("utf-8").splitlines()]
has_json = sum(1 for record in predictions
               if re.search(r"\{.*\}", record["raw_output"], re.DOTALL))

print(f"answers containing a JSON object: {has_json}/{len(predictions)}")
print(f"mean generation time: "
      f"{sum(r['generation_seconds'] for r in predictions) / len(predictions):.1f} s")


answers containing a JSON object: 200/200
mean generation time: 34.5 s


## 13. Back in the repo

```powershell
cd D:\it_project\github_sync\zip-vlm\linkedin-zip-challenge
# download p4c_holdout_predictions.jsonl from Drive first
uv run python -m src.core.vl_models.score_predictions <path>\p4c_holdout_predictions.jsonl
```

That prints the four metric layers plus a per-wall-count breakdown, and writes a JSON
artifact under `ai-collab/reports/artifacts/vl-p4c/`.

**Read `exact_match` first, not wall F1.** A board is only usable when every wall is
right, and the two numbers come apart badly: at six walls per board a per-wall accuracy
of 0.85 leaves only 38% of boards fully correct. Reference points:

| measurement | number |
|---|---|
| P4a, 4 held-out samples, 50 steps | wall F1 0.958 |
| untuned `qwen3.5:4b-q8_0`, six real screenshots | cell 0.961, wall F1 0.438, end-to-end 2/6 |

The first is the bar to clear, on fifty times as many samples. **The second is not
comparable** - different images, different prompt, different difficulty - and quoting it
as though this run beat it would be exactly the domain-gap claim that the deferred P3
was the only way to support.

## When this notebook is finished

`Runtime -> Disconnect and delete runtime`. The runtime bills at the same rate whether it
is training or idle (measured 2026-08-22: L4 = 1.54 compute units/hour), so a session left
connected overnight costs more than this entire stage.
